# Analyse exploratoire — maturité digitale des entreprises françaises

Ce notebook répond aux trois questions posées dans le README, à partir de la table de scores produite par `src/03_sql_transformations.py` :

1. Quels secteurs sont les plus avancés dans le numérique ?
2. Les petites entreprises sont-elles en retard par rapport aux grandes ?
3. Sur quel pilier (cloud, IA, données) le retard est-il le plus fort ?

**La visualisation n'est pas ici** : c'est le rôle du tableau de bord Power BI. Ce notebook sert à établir les chiffres et à expliciter la méthode, notamment le piège de structure décrit juste en dessous.

## Précaution de méthode : les 65 lignes ne sont pas 65 observations

Deux pièges se cumulent dans ces données, et les ignorer donne des résultats faux sans aucun message d'erreur.

**1. Certaines lignes sont des totaux d'autres lignes.** Pour un secteur donné, la ligne `Ens` est la somme de ses tranches `po1..po4`. Et `S_TIC` est libellé par l'INSEE « *dont* secteur producteur de TIC » : ses entreprises sont déjà comptées dans l'industrie et dans l'information-communication. Les colonnes `est_agregat` et `est_sous_ensemble` permettent de les écarter.

**2. Les sous-scores sont des taux, pas des effectifs.** Additionner ou moyenner `score_cloud` sur plusieurs lignes revient à mélanger des pourcentages calculés sur des populations différentes — y compris après avoir écarté les agrégats.

Conséquence sur la méthode suivie ici : **pour chaque question, on utilise la ligne déjà agrégée par l'INSEE au bon niveau**, plutôt que de recombiner les cellules soi-même. Pour classer les secteurs, on lit les lignes `taille == 'Ens'`, qui sont les totaux sectoriels publiés. Pour comparer les tailles, on lit les lignes `secteur == 'Ens'`.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)

# Racine du depot trouvee automatiquement : le notebook fonctionne quel que
# soit le dossier de travail du noyau.
RACINE = Path.cwd()
while not (RACINE / 'data' / 'processed').is_dir() and RACINE != RACINE.parent:
    RACINE = RACINE.parent

scores = pd.read_csv(RACINE / 'data' / 'processed' / 'entreprise_score_final.csv')
libelles = pd.read_csv(RACINE / 'data' / 'reference' / 'metadonnees_irecotic23.csv', sep=';')

# Dictionnaire officiel des codes, pour afficher des noms lisibles.
noms_secteur = dict(
    libelles.loc[libelles['COD_VAR'] == 'ACT_IR', ['COD_MOD', 'LIB_MOD1']].itertuples(index=False)
)
noms_taille = dict(
    libelles.loc[libelles['COD_VAR'] == 'Taille_IR',
                 ['COD_MOD', 'LIB_MOD1']].itertuples(index=False)
)

print(f'{len(scores)} lignes chargees')
scores.head(3)

65 lignes chargees


,secteur,taille,total_entreprises,score_cloud,score_ia,score_data,pct_erp_utilisation,digital_maturity_score,secret_statistique,est_agregat,est_sous_ensemble
0,S_M,po4,353,26.18,14.73,20.99,96.88,61.90,0,0,0
1,S_J_G95.1,po4,355,24.00,13.75,23.83,95.21,61.58,0,0,0
2,S_TIC,po4,355,23.49,13.75,24.08,96.34,61.32,0,0,1


### Vérification du piège, en chiffres

Avant d'analyser quoi que ce soit, on mesure l'ampleur du double comptage. C'est ce qui justifie toute la suite.

In [2]:
decompte = {
    'lignes totales': len(scores),
    'dont agregats (Ens)': int(scores['est_agregat'].sum()),
    'dont sous-ensemble (S_TIC)': int(scores['est_sous_ensemble'].sum()),
}

disjointes = scores[(scores['est_agregat'] == 0) & (scores['est_sous_ensemble'] == 0)]
decompte['cellules disjointes'] = len(disjointes)

national = scores[(scores['secteur'] == 'Ens') & (scores['taille'] == 'Ens')].iloc[0]
population_reelle = int(national['total_entreprises'])

for cle, valeur in decompte.items():
    print(f'{cle:30} {valeur:>8}')
print()
total_brut = int(scores['total_entreprises'].sum())
print(f"{'somme sur les 65 lignes':30} {total_brut:>8}  <- double comptage")
print(f"{'somme sur les disjointes':30} {int(disjointes['total_entreprises'].sum()):>8}")
print(f"{'population reelle':30} {population_reelle:>8}  <- ligne Ens / Ens")

lignes totales                       65
dont agregats (Ens)                  17
dont sous-ensemble (S_TIC)            5
cellules disjointes                  44

somme sur les 65 lignes          704384  <- double comptage
somme sur les disjointes         172348
population reelle                172345  <- ligne Ens / Ens


## Question 1 — quels secteurs sont les plus avancés ?

On lit les lignes `taille == 'Ens'`, c'est-à-dire le total publié pour chaque secteur, en écartant `Ens` (le total France, qui sert de repère) et `S_TIC` (sous-ensemble).

In [3]:
repere = national['digital_maturity_score']

par_secteur = (
    scores[(scores['taille'] == 'Ens')
           & (scores['secteur'] != 'Ens')
           & (scores['est_sous_ensemble'] == 0)]
    .assign(libelle=lambda d: d['secteur'].map(noms_secteur))
    .loc[:, ['secteur', 'libelle', 'total_entreprises', 'score_cloud', 'score_ia',
             'score_data', 'digital_maturity_score']]
    .sort_values('digital_maturity_score', ascending=False)
    .reset_index(drop=True)
)
par_secteur['ecart_au_repere'] = (par_secteur['digital_maturity_score'] - repere).round(2)

print(f'Repere national (Ens / Ens) : {repere} / 100')
amplitude = (par_secteur['digital_maturity_score'].iloc[0]
             - par_secteur['digital_maturity_score'].iloc[-1])
print(f'Amplitude entre le premier et le dernier secteur : {amplitude:.2f} points')
par_secteur

Repere national (Ens / Ens) : 18.99 / 100
Amplitude entre le premier et le dernier secteur : 35.61 points


,secteur,libelle,total_entreprises,score_cloud,score_ia,score_data,digital_maturity_score,ecart_au_repere
0,S_J_G95.1,"Information et communication, réparation d'ord...",7816,20.44,11.65,15.54,47.63,28.64
1,S_M,"Activités spécialisées, scientifiques et techn...",17860,13.56,5.50,9.71,28.77,9.78
2,S_L,Activités immobilières,2796,13.59,2.80,10.57,26.97,7.98
3,S_G_D45-46,Commerce de gros et réparation d'automobiles e...,18572,9.40,2.11,12.63,24.15,5.16
4,S_DE,"Énergie, eau, gestion déchets et dépollution",1280,9.14,2.00,12.56,23.70,4.71
5,S_C,Industrie manufacturière,26571,7.20,1.95,9.80,18.95,-0.04
6,S_N,Activités de services administratifs et de sou...,15966,8.08,2.24,8.00,18.32,-0.67
7,S_G_D47,Commerce de détail,19884,4.79,1.18,8.27,14.24,-4.75
8,S_H,Transports et entreposage,10811,5.63,0.94,6.34,12.92,-6.07
9,S_I,Hébergement et restauration,21559,4.89,0.99,6.31,12.19,-6.80


## Question 2 — les petites entreprises sont-elles en retard ?

On lit les lignes `secteur == 'Ens'`, c'est-à-dire le total publié pour chaque tranche de taille.

In [4]:
par_taille = (
    scores[(scores['secteur'] == 'Ens') & (scores['taille'] != 'Ens')]
    .assign(libelle=lambda d: d['taille'].map(lambda t: noms_taille.get(t.upper(), t)))
    .loc[:, ['taille', 'libelle', 'total_entreprises', 'score_cloud', 'score_ia',
             'score_data', 'digital_maturity_score']]
    .sort_values('taille')
    .reset_index(drop=True)
)
rapport = (par_taille['digital_maturity_score'].iloc[-1]
           / par_taille['digital_maturity_score'].iloc[0])

print(f'Rapport grandes / petites sur le score global : {rapport:.1f}x')
par_taille

Rapport grandes / petites sur le score global : 3.6x


,taille,libelle,total_entreprises,score_cloud,score_ia,score_data,digital_maturity_score
0,po1,Moins de 20 personnes occupées,94858,6.13,1.69,5.74,13.56
1,po2,De 20 à 49 personnes occupées,50237,8.48,2.26,10.43,21.18
2,po3,De 50 à 249 personnes occupées,22321,12.37,4.06,14.00,30.43
3,po4,250 personnes occupées ou plus,4929,20.17,8.37,20.79,49.34


### Secteur ou taille : qui discrimine le plus ?

Les deux questions précédentes se répondent séparément, mais la comparaison des deux dimensions demande de les mettre sur la meme unité. On prend l'amplitude en points de score, chacune a son niveau d'agregation propre.

In [5]:
# Secteur ou taille : laquelle des deux dimensions discrimine le plus ?
# On compare sur la meme unite, en points de score, chacune a son propre
# niveau d'agregation INSEE.
amplitudes = pd.DataFrame([
    {
        'dimension': 'secteur (a taille Ens)',
        'minimum': par_secteur['digital_maturity_score'].min(),
        'maximum': par_secteur['digital_maturity_score'].max(),
    },
    {
        'dimension': 'taille (a secteur Ens)',
        'minimum': par_taille['digital_maturity_score'].min(),
        'maximum': par_taille['digital_maturity_score'].max(),
    },
])
amplitudes['amplitude_points'] = (amplitudes['maximum'] - amplitudes['minimum']).round(2)
amplitudes['rapport_max_sur_min'] = (amplitudes['maximum'] / amplitudes['minimum']).round(1)
amplitudes

,dimension,minimum,maximum,amplitude_points,rapport_max_sur_min
0,secteur (a taille Ens),12.02,47.63,35.61,4.0
1,taille (a secteur Ens),13.56,49.34,35.78,3.6


## Question 3 — sur quel pilier le retard est-il le plus fort ?

Les trois piliers n'ont pas le même poids (cloud 30, IA 40, données 30), donc leurs scores ne sont pas comparables directement. On ramène chacun à son **taux d'adoption** : la part des points effectivement obtenus sur les points disponibles. Un taux de 50 % signifie qu'une entreprise sur deux du périmètre est concernée.

In [6]:
PONDERATIONS = {'score_cloud': 30, 'score_ia': 40, 'score_data': 30}

remplissage = pd.DataFrame(
    [
        {
            'pilier': pilier.replace('score_', ''),
            'points_obtenus': national[pilier],
            'points_disponibles': poids,
            'taux_adoption_pct': round(national[pilier] / poids * 100, 1),
        }
        for pilier, poids in PONDERATIONS.items()
    ]
).sort_values('taux_adoption_pct')

print('Au niveau national (ligne Ens / Ens) :')
remplissage

Au niveau national (ligne Ens / Ens) :


,pilier,points_obtenus,points_disponibles,taux_adoption_pct
1,ia,2.35,40,5.9
0,cloud,8.03,30,26.8
2,data,8.61,30,28.7


In [7]:
# Le meme calcul par taille, pour voir si le retard se joue sur le meme pilier
# selon la taille d'entreprise.
taux = par_taille.set_index('libelle')[list(PONDERATIONS)].div(pd.Series(PONDERATIONS)) * 100
taux.columns = [c.replace('score_', 'taux_') for c in taux.columns]
taux.round(1)

,taux_cloud,taux_ia,taux_data
libelle,,,
Moins de 20 personnes occupées,20.4,4.2,19.1
De 20 à 49 personnes occupées,28.3,5.6,34.8
De 50 à 249 personnes occupées,41.2,10.1,46.7
250 personnes occupées ou plus,67.2,20.9,69.3


## Le secret statistique

Quand l'effectif est trop faible pour être publié sans risque de réidentification, l'INSEE masque la case (valeur `s`). Ces cases sont chargées comme valeurs nulles et **jamais comme des 0** : un 0 serait une mesure, alors qu'il n'y a pas de mesure. Le score global devient donc incalculable pour ces lignes, ce que signale `secret_statistique`.

In [8]:
masquees = scores[scores['secret_statistique'] == 1].assign(
    libelle=lambda d: d['secteur'].map(noms_secteur)
)
masquees[['secteur', 'libelle', 'taille', 'total_entreprises', 'score_cloud',
          'score_ia', 'score_data', 'digital_maturity_score']]

,secteur,libelle,taille,total_entreprises,score_cloud,score_ia,score_data,digital_maturity_score
61,S_DE,"Énergie, eau, gestion déchets et dépollution",po2,458,7.14,NaN,10.68,NaN
62,S_DE,"Énergie, eau, gestion déchets et dépollution",po4,72,22.50,NaN,26.67,NaN
63,S_L,Activités immobilières,po2,716,11.10,NaN,10.06,NaN
64,S_L,Activités immobilières,po4,110,21.55,NaN,24.27,NaN


## Ce qu'il faut retenir

- **Secteur et taille pèsent le même poids.** C'est le résultat le plus net de cette analyse, et il est contre-intuitif : l'amplitude est de 35,6 points entre le secteur le mieux et le moins bien classé, et de 35,8 points entre les plus petites et les plus grandes entreprises. Aucune des deux dimensions ne domine l'autre. Un petit acteur de l'information-communication et une grande entreprise de la construction peuvent se retrouver au même niveau de maturité.
- **L'IA est le pilier le plus en retard**, et de loin : 5,9 % d'adoption au niveau national, contre 26,8 % pour le cloud et 28,7 % pour l'analyse de données. Comme c'est aussi le pilier le plus lourdement pondéré (40 points sur 100), il tire mécaniquement tous les scores vers le bas — ce qui explique qu'aucune ligne ne dépasse 61,9 sur 100.
- **Le retard en IA ne se rattrape pas avec la taille.** Même chez les entreprises de 250 personnes et plus, l'adoption de l'IA reste à 20,9 %, quand le cloud et les données dépassent 67 %. L'écart entre piliers est structurel, pas un simple effet de taille.
- **Le score n'est pas une note absolue.** Les pondérations 30/40/30 sont un choix du projet ; l'échelle réellement parcourue par les données va de 6,6 à 61,9. Il sert à comparer des lignes entre elles, pas à dire si un secteur est « bon ».
- **Quatre lignes n'ont pas de score**, par secret statistique sur l'indicateur IA. C'est une information, pas un trou à combler par un zéro.